# 📚 Aggregation & GroupBy
### 집계 & GroupBy

> **Section 7 of 11** · Pandas Complete Reference Guide for a JS/TS developer transitioning into BA  
> 전체 11개 섹션 중 **7번째** · JS/TS 개발자 출신 BA를 위한 Pandas 완전 참조 가이드

---
# 🎯 Learning Objective
Today I want to learn: / 오늘 배울 내용:
- [x] How to compute basic aggregates and group data by one or more keys, always finishing with `reset_index()`  
기본 집계를 계산하고 하나 이상의 기준으로 그룹화한 뒤, 항상 `reset_index()`로 마무리하는 방법
- [x] How named aggregation (`agg` dict) applies a different function per column, and how `transform()` / `filter()` differ from a plain `agg()`  
named aggregation(`agg` dict)이 열마다 다른 함수를 적용하는 방법, `transform()` / `filter()`가 일반 `agg()`와 다른 점
- [x] How to compare across groups over time with `shift()` / `expanding()`, find a group's extreme row with `idxmax()` / `idxmin()`, and group by time with `pd.Grouper`  
`shift()` / `expanding()`로 시간에 따라 그룹을 비교하고, `idxmax()` / `idxmin()`으로 그룹의 극값 행을 찾고, `pd.Grouper`로 시간 기준 그룹화하는 방법

---
# 🧠 Concept

## What is it?
*(Explain it in your own words.)*

**English**
GroupBy is pandas' answer to "break this into groups, then summarize each group" — the same shape as a SQL `GROUP BY` or a JS `arr.reduce()` that builds a frequency/summary object. It's a 3-step mental model every time: **split** the data into groups by one or more keys, **apply** a function to each group, and **combine** the results back into one table.

**한글**
GroupBy는 "이걸 그룹으로 나눈 뒤, 각 그룹을 요약하라"는 요청에 대한 pandas의 답입니다 — SQL의 `GROUP BY`나, 빈도·요약 객체를 만드는 JS의 `arr.reduce()`와 같은 형태입니다. 매번 3단계로 생각하면 됩니다: 하나 이상의 기준으로 데이터를 그룹으로 **분할**(split)하고, 각 그룹에 함수를 **적용**(apply)한 뒤, 결과를 다시 하나의 테이블로 **결합**(combine)합니다.

## Why do we use it?
*(When is it useful?)*

**English**
Almost no business question is about a single row — it's "total *by* region," "average *by* department," "top order *in each* category." Without GroupBy you'd write a manual loop that filters, computes, and appends for every unique group value one at a time; GroupBy does all three steps in one vectorized call.

**한글**
비즈니스 질문 중 단일 행에 관한 것은 거의 없습니다 — "지역*별* 합계", "부서*별* 평균", "카테고리*마다* 최대 주문"입니다. GroupBy가 없다면 고유한 그룹 값마다 하나씩 필터링하고, 계산하고, 추가하는 반복문을 직접 작성해야 합니다. GroupBy는 이 세 단계를 한 번의 벡터 연산 호출로 처리합니다.

## When is it used in Business Analytics?
*(Real-world use case)*

**English**
This is arguably the single most business-relevant chapter in this whole guide — "monthly revenue by product line," "headcount by department," "conversion rate by channel" are all one `groupby().agg()` call. Nearly every dashboard metric traces back to a GroupBy.

**한글**
이는 이 가이드 전체에서 비즈니스와 가장 밀접한 챕터라고 해도 과언이 아닙니다 — "제품 라인별 월 매출", "부서별 인원", "채널별 전환율"은 전부 한 번의 `groupby().agg()` 호출입니다. 거의 모든 대시보드 지표가 결국 GroupBy로 거슬러 올라갑니다.

### Quick Comparison: JS/TS vs pandas / 빠른 비교

| Concept / 개념 | JavaScript / TypeScript | pandas |
|---|---|---|
| Group + summarize / 그룹화 + 요약 | `arr.reduce()` building a summary object / 요약 객체를 만드는 `arr.reduce()` | `df.groupby("key").agg(...)` |
| Sum within each group / 그룹별 합계 | manual accumulator object / 직접 만든 누산 객체 | `df.groupby("key")["col"].sum()` |
| Different summary per field / 필드마다 다른 요약 | manual object construction / 직접 객체 구성 | named aggregation: `.agg(x=("col","sum"), ...)` |
| Broadcast a group total back to every row / 그룹 합계를 모든 행에 다시 broadcast | a second pass with a lookup map / 조회 맵으로 두 번째 순회 | `.transform("sum")` |
| Keep only groups matching a condition / 조건을 만족하는 그룹만 유지 | `Object.entries` + `filter` + flatten | `.filter(lambda g: ...)` |
| Compare to the previous period / 이전 기간과 비교 | manual index-1 lookup / 직접 index-1 조회 | `.shift(1)` |

---
# 📝 Syntax

## Basic Syntax

In [1]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["Seoul", "Busan", "Seoul", "Incheon"],
    "sales": [320000, 85000, 150000, 43000],
})

# split -> apply -> combine, in one line / split -> apply -> combine를 한 줄로
print(orders.groupby("region")["sales"].sum())
print()

# Almost always followed by reset_index() to get a normal DataFrame back
# 거의 항상 reset_index()를 붙여서 일반 DataFrame으로 되돌림
print(orders.groupby("region")["sales"].sum().reset_index())

region
Busan       85000
Incheon     43000
Seoul      470000
Name: sales, dtype: int64

    region   sales
0    Busan   85000
1  Incheon   43000
2    Seoul  470000


## Common Variations

In [2]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["Seoul", "Busan", "Seoul", "Incheon"],
    "category": ["Elec", "Cloth", "Cloth", "Elec"],
    "sales": [320000, 85000, 150000, 43000],
})

# Group by more than one key / 여러 기준으로 그룹화
print(orders.groupby(["region", "category"])["sales"].sum().reset_index())
print()

# Named aggregation -- a different function per column, all in one call
# named aggregation -- 열마다 다른 함수를, 한 번의 호출로
print(orders.groupby("region").agg(total=("sales", "sum"), count=("sales", "count")).reset_index())

    region category   sales
0    Busan    Cloth   85000
1  Incheon     Elec   43000
2    Seoul    Cloth  150000
3    Seoul     Elec  320000

    region   total  count
0    Busan   85000      1
1  Incheon   43000      1
2    Seoul  470000      2


---
# 🧪 Small Examples

## Example 1 — Basic Aggregation Functions
*(Covers source section 7-1)*

**English:** `.sum()` / `.mean()` / `.std()` / `.min()` / `.max()` work exactly like their JS equivalents, applied to a whole column at once. `.count()` and `.size` look similar but differ in one important way: `.count()` skips `NaN` values, while `.size` counts every row regardless. `.agg([...])` runs several functions on multiple columns in a single call.  
**한글:** `.sum()` / `.mean()` / `.std()` / `.min()` / `.max()`는 JS의 대응 함수와 똑같이 동작하며, 열 전체에 한 번에 적용됩니다. `.count()`와 `.size`는 비슷해 보이지만 중요한 차이가 있습니다: `.count()`는 `NaN`을 건너뛰고, `.size`는 결측 여부와 상관없이 모든 행을 셉니다. `.agg([...])`는 한 번의 호출로 여러 열에 여러 함수를 실행합니다.

In [3]:
import pandas as pd
import numpy as np

orders = pd.DataFrame({
    "region": ["Seoul","Busan","Seoul","Incheon","Seoul","Busan","Incheon","Seoul","Busan","Seoul"],
    "sales": [320000, 85000, 150000, 43000, 210000, 95000, 38000, 67000, None, 190000],
    "quantity": [2, 3, 1, 5, 2, 1, 4, 3, 2, 1],
})

print("sum   :", orders["sales"].sum())
print("mean  :", orders["sales"].mean())
print("count :", orders["sales"].count())   # skips the one NaN -> 9 / NaN 하나를 건너뜀 -> 9
print("size  :", orders["sales"].size)      # counts every row -> 10 / 모든 행을 셈 -> 10
print("std   :", orders["sales"].std())
print("min/max:", orders["sales"].min(), "/", orders["sales"].max())
print()

# agg([...]) -- several functions, several columns, one call
# agg([...]) -- 여러 함수, 여러 열, 한 번의 호출
print(orders[["sales", "quantity"]].agg(["sum", "mean", "count"]))

sum   : 1198000.0
mean  : 133111.11111111112
count : 9
size  : 10
std   : 93317.26052082279
min/max: 38000.0 / 320000.0

              sales  quantity
sum    1.198000e+06      24.0
mean   1.331111e+05       2.4
count  9.000000e+00      10.0


## Example 2 — GroupBy Basics: Single & Multiple Keys
*(Covers source section 7-2)*

**English:** `df.groupby("key")["col"].sum()` groups by a single column. Passing a list — `df.groupby(["key1","key2"])` — groups by the combination of both. Either way, the result's index is the group key(s), not a normal `RangeIndex` — which is why `.reset_index()` almost always follows.  
**한글:** `df.groupby("key")["col"].sum()`은 열 하나를 기준으로 그룹화합니다. 리스트를 전달하면(`df.groupby(["key1","key2"])`) 두 열의 조합을 기준으로 그룹화합니다. 어느 쪽이든 결과의 인덱스는 일반 `RangeIndex`가 아니라 그룹 키가 되는데, 그래서 거의 항상 `.reset_index()`가 뒤따릅니다.

In [4]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": [1,2,3,4,5,6,7,8,9,10],
    "region": ["Seoul","Busan","Seoul","Incheon","Seoul","Busan","Incheon","Seoul","Busan","Seoul"],
    "category": ["Elec","Cloth","Elec","Food","Cloth","Elec","Cloth","Food","Elec","Elec"],
    "sales": [320000, 85000, 150000, 43000, 210000, 95000, 38000, 67000, None, 190000],
})

# Single key / 단일 기준
print("groupby('region') -- notice the index is 'region', not a number:")
print(orders.groupby("region")["sales"].sum())
print()

print("...and after reset_index(), it's a normal column again:")
print(orders.groupby("region")["sales"].sum().reset_index())
print()

# Multiple keys / 복수 기준
print("groupby(['region','category']):")
print(orders.groupby(["region", "category"])["sales"].sum().reset_index())

groupby('region') -- notice the index is 'region', not a number:
region
Busan      180000.0
Incheon     81000.0
Seoul      937000.0
Name: sales, dtype: float64

...and after reset_index(), it's a normal column again:
    region     sales
0    Busan  180000.0
1  Incheon   81000.0
2    Seoul  937000.0

groupby(['region','category']):
    region category     sales
0    Busan    Cloth   85000.0
1    Busan     Elec   95000.0
2  Incheon    Cloth   38000.0
3  Incheon     Food   43000.0
4    Seoul    Cloth  210000.0
5    Seoul     Elec  660000.0
6    Seoul     Food   67000.0


## Example 3 — Named Aggregation: The agg dict Pattern
*(Covers source section 7-3)*

**English:** This is the single most-used real-world GroupBy pattern: `.agg(new_name=("source_col", "function"))` computes several *different* statistics from *different* columns in one call, and names the result columns exactly what you want — no `.rename()` needed afterward.  
**한글:** 이는 실무에서 가장 많이 쓰이는 GroupBy 패턴입니다: `.agg(새이름=("원본열", "함수"))`는 한 번의 호출로 *서로 다른* 열에서 *서로 다른* 통계를 계산하고, 결과 열의 이름을 원하는 대로 정확히 붙입니다 — 이후 `.rename()`이 필요 없습니다.

In [5]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": [1,2,3,4,5,6,7,8,9,10],
    "region": ["Seoul","Busan","Seoul","Incheon","Seoul","Busan","Incheon","Seoul","Busan","Seoul"],
    "sales": [320000, 85000, 150000, 43000, 210000, 95000, 38000, 67000, None, 190000],
    "quantity": [2, 3, 1, 5, 2, 1, 4, 3, 2, 1],
})

result = orders.groupby("region").agg(
    total_sales = ("sales", "sum"),
    avg_sales   = ("sales", "mean"),
    order_count = ("order_id", "count"),
    total_qty   = ("quantity", "sum"),
).reset_index()
print(result)

    region  total_sales  avg_sales  order_count  total_qty
0    Busan     180000.0    90000.0            3          6
1  Incheon      81000.0    40500.0            2          9
2    Seoul     937000.0   187400.0            5          9


## Example 4 — transform: Broadcasting a Group Value Back
*(Covers source section 7-4)*

**English:** `groupby().agg()` **shrinks** to one row per group. `groupby().transform()` does the opposite — it keeps the **original row count**, repeating each group's aggregate value on every row that belongs to it. This is exactly what you need for "this row's share of its group's total" or "this row's rank within its group."  
**한글:** `groupby().agg()`는 그룹당 한 행으로 **줄어듭니다**. `groupby().transform()`은 반대입니다 — **원본 행 개수를 그대로 유지**하면서, 각 그룹의 집계값을 그 그룹에 속한 모든 행에 반복해서 붙입니다. "이 행이 그룹 합계에서 차지하는 비중"이나 "그룹 내 이 행의 순위"를 구할 때 정확히 필요한 방식입니다.

In [6]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": [1,2,3,4,5,6,7,8,9,10],
    "region": ["Seoul","Busan","Seoul","Incheon","Seoul","Busan","Incheon","Seoul","Busan","Seoul"],
    "sales": [320000, 85000, 150000, 43000, 210000, 95000, 38000, 67000, None, 190000],
})

# transform -- same row count as the original (10), unlike agg's shrunk result
# transform -- agg의 축소된 결과와 달리, 원본과 같은 행 개수(10) 유지
orders["region_total"] = orders.groupby("region")["sales"].transform("sum")
orders["share_pct"] = (orders["sales"] / orders["region_total"] * 100).round(1)
orders["rank_in_region"] = orders.groupby("region")["sales"].rank(ascending=False, method="min")

print(orders[["region", "sales", "region_total", "share_pct", "rank_in_region"]].sort_values(["region", "rank_in_region"]))

    region     sales  region_total  share_pct  rank_in_region
5    Busan   95000.0      180000.0       52.8             1.0
1    Busan   85000.0      180000.0       47.2             2.0
8    Busan       NaN      180000.0        NaN             NaN
3  Incheon   43000.0       81000.0       53.1             1.0
6  Incheon   38000.0       81000.0       46.9             2.0
0    Seoul  320000.0      937000.0       34.2             1.0
4    Seoul  210000.0      937000.0       22.4             2.0
9    Seoul  190000.0      937000.0       20.3             3.0
2    Seoul  150000.0      937000.0       16.0             4.0
7    Seoul   67000.0      937000.0        7.2             5.0


## Example 5 — filter: Keeping or Dropping Whole Groups
*(Covers source section 5-5... continued here as 7-5)*

**English:** `groupby().filter(lambda g: condition)` evaluates the condition **once per group** — if it's `True`, *every row* in that group survives; if `False`, the whole group is dropped. Unlike `agg()`, the row count only shrinks if entire groups are removed — surviving rows keep their original shape.  
**한글:** `groupby().filter(lambda g: condition)`은 조건을 **그룹마다 한 번씩** 평가합니다 — `True`면 그 그룹의 *모든 행*이 살아남고, `False`면 그룹 전체가 제거됩니다. `agg()`와 달리, 행 개수는 그룹 전체가 제거될 때만 줄어들며 — 살아남은 행은 원래 모습 그대로 유지됩니다.

In [7]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": [1,2,3,4,5,6,7,8,9,10],
    "region": ["Seoul","Busan","Seoul","Incheon","Seoul","Busan","Incheon","Seoul","Busan","Seoul"],
    "sales": [320000, 85000, 150000, 43000, 210000, 95000, 38000, 67000, None, 190000],
})

# Keep only regions whose AVERAGE sale is >= 100,000 / 평균 매출이 100,000 이상인 지역만 유지
print("regions averaging >= 100,000 (Seoul only -- Busan and Incheon are both below):")
print(orders.groupby("region").filter(lambda x: x["sales"].mean() >= 100000))
print()

# Keep only regions with at least 3 orders / 주문이 3건 이상인 지역만 유지
print("regions with 3+ orders (Incheon, with only 2, is excluded):")
print(orders.groupby("region").filter(lambda x: len(x) >= 3))

regions averaging >= 100,000 (Seoul only -- Busan and Incheon are both below):
   order_id region     sales
0         1  Seoul  320000.0
2         3  Seoul  150000.0
4         5  Seoul  210000.0
7         8  Seoul   67000.0
9        10  Seoul  190000.0

regions with 3+ orders (Incheon, with only 2, is excluded):
   order_id region     sales
0         1  Seoul  320000.0
1         2  Busan   85000.0
2         3  Seoul  150000.0
4         5  Seoul  210000.0
5         6  Busan   95000.0
7         8  Seoul   67000.0
8         9  Busan       NaN
9        10  Seoul  190000.0


## Example 6 — shift & expanding: Sequential Group Calculations
*(Covers source sections 7-6 and 7-7)*

**English:** `.shift(1)` pulls the *previous row's* value into the current row — the standard way to compute month-over-month change; combined with `groupby()`, each group's "previous" only ever looks within that same group. `.expanding()` runs a *cumulative* aggregate from the very first row up to the current one (a running total, a running max), unlike `.rolling(n)`'s fixed-size window from Section 10.  
**한글:** `.shift(1)`은 *이전 행*의 값을 현재 행으로 가져옵니다 — 전월 대비 변화를 계산하는 표준 방법입니다. `groupby()`와 결합하면, 각 그룹의 "이전"은 항상 같은 그룹 안에서만 찾습니다. `.expanding()`은 첫 행부터 현재 행까지 *누적* 집계를 실행합니다(누적 합계, 누적 최댓값) — 10번 섹션에서 다룰 `.rolling(n)`의 고정 크기 윈도우와는 다릅니다.

In [8]:
import pandas as pd

# shift -- month-over-month change / shift -- 전월 대비 변화
monthly = pd.DataFrame({
    "month": ["2024-01","2024-02","2024-03","2024-04","2024-05","2024-06"],
    "sales": [5678000, 5188000, 5560000, 5766000, 5431000, 5325000],
})
monthly["MoM_pct"] = (monthly["sales"].pct_change() * 100).round(1)
print("shift-based MoM (whole company):")
print(monthly)
print()

# groupby + shift -- MoM calculated SEPARATELY within each region
# groupby + shift -- 각 지역 안에서 따로 계산되는 MoM
monthly_region = pd.DataFrame({
    "month": ["2024-01","2024-01","2024-02","2024-02","2024-03","2024-03"],
    "region": ["Seoul","Busan","Seoul","Busan","Seoul","Busan"],
    "sales": [1200000, 800000, 1450000, 950000, 980000, 720000],
})
monthly_region["prev"] = monthly_region.groupby("region")["sales"].shift(1)
monthly_region["MoM_pct"] = ((monthly_region["sales"] / monthly_region["prev"] - 1) * 100).round(1)
print("groupby + shift -- per-region MoM:")
print(monthly_region)
print()

# expanding -- cumulative from the very first row / expanding -- 첫 행부터의 누적
monthly["cumsum"] = monthly["sales"].expanding().sum()
monthly["cummax"] = monthly["sales"].expanding().max()
print("expanding -- running total and running max:")
print(monthly[["month", "sales", "cumsum", "cummax"]])

shift-based MoM (whole company):
     month    sales  MoM_pct
0  2024-01  5678000      NaN
1  2024-02  5188000     -8.6
2  2024-03  5560000      7.2
3  2024-04  5766000      3.7
4  2024-05  5431000     -5.8
5  2024-06  5325000     -2.0

groupby + shift -- per-region MoM:
     month region    sales       prev  MoM_pct
0  2024-01  Seoul  1200000        NaN      NaN
1  2024-01  Busan   800000        NaN      NaN
2  2024-02  Seoul  1450000  1200000.0     20.8
3  2024-02  Busan   950000   800000.0     18.8
4  2024-03  Seoul   980000  1450000.0    -32.4
5  2024-03  Busan   720000   950000.0    -24.2

expanding -- running total and running max:
     month    sales      cumsum     cummax
0  2024-01  5678000   5678000.0  5678000.0
1  2024-02  5188000  10866000.0  5678000.0
2  2024-03  5560000  16426000.0  5678000.0
3  2024-04  5766000  22192000.0  5766000.0
4  2024-05  5431000  27623000.0  5766000.0
5  2024-06  5325000  32948000.0  5766000.0


## Example 7 — idxmax / idxmin & pd.Grouper: Specialized GroupBy Tools
*(Covers source sections 7-8 and 7-9)*

**English:** `.idxmax()` / `.idxmin()` return the **row index** where a value is largest/smallest — not the value itself — which is exactly what's needed to pull the *entire row* (all its other columns too) via `.loc[]`. Combined with `groupby()`, it finds each group's top/bottom row in one line. `pd.Grouper(freq="ME")` groups by a time period (monthly, weekly, quarterly) and can be combined with an ordinary column for "monthly totals, per region" in a single `groupby()` call.  
**한글:** `.idxmax()` / `.idxmin()`은 값 자체가 아니라 값이 가장 크거나 작은 **행의 인덱스**를 반환합니다 — 이는 `.loc[]`로 *행 전체*(다른 모든 열까지)를 꺼내는 데 정확히 필요한 것입니다. `groupby()`와 결합하면 한 줄로 각 그룹의 최고/최저 행을 찾을 수 있습니다. `pd.Grouper(freq="ME")`는 시간 단위(월별, 주별, 분기별)로 그룹화하며, 일반 열과 결합해서 "지역별 월별 합계"를 한 번의 `groupby()` 호출로 구할 수 있습니다.

In [ ]:
import pandas as pd

# idxmax -- the ROW INDEX of the max, not the max value itself
# idxmax -- 최댓값 자체가 아니라 최댓값의 행 인덱스
orders = pd.DataFrame({
    "region": ["Seoul", "Busan", "Seoul", "Incheon", "Seoul", "Busan"],
    "product": ["Laptop", "Mouse", "Keyboard", "Monitor", "Webcam", "Laptop"],
    "sales": [320000, 85000, 150000, 43000, 210000, 95000],
})
top_idx = orders.groupby("region")["sales"].idxmax()
print("row index of each region's top sale:")
print(top_idx)
print()
print("the FULL rows behind those indexes:")
print(orders.loc[top_idx])
print()

# pd.Grouper -- group by a time frequency, needs a DatetimeIndex first
# pd.Grouper -- 시간 빈도로 그룹화, 먼저 DatetimeIndex가 필요함
np.random.seed(1)
daily = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=90, freq="D"),
    "region": ["Seoul", "Busan", "Incheon"] * 30,
    "sales": np.random.randint(100000, 500000, 90),
})
monthly_by_region = (
    daily
    .set_index("date")
    .groupby([pd.Grouper(freq="ME"), "region"])["sales"]
    .sum()
    .reset_index()
)
print("pd.Grouper(freq='ME') + region -- monthly totals per region:")
print(monthly_by_region.head(6))

row index of each region's top sale:
region
Busan      5
Incheon    3
Seoul      0
Name: sales, dtype: int64

the FULL rows behind those indexes:
    region  product   sales
5    Busan   Laptop   95000
3  Incheon  Monitor   43000
0    Seoul   Laptop  320000

pd.Grouper(freq='ME') + region -- monthly totals per region:
        date   region    sales
0 2024-01-31    Busan  2818214
1 2024-01-31  Incheon  3063152
2 2024-01-31    Seoul  3222252
3 2024-02-29    Busan  3415879
4 2024-02-29  Incheon  3750416
5 2024-02-29    Seoul  2322270


## Example 8 — MultiIndex Basics: What GroupBy Produces
*(Covers source section 7-10)*

**English:** Grouping by more than one key produces a **MultiIndex** — an index with two (or more) levels stacked together. `.loc["Seoul"]` pulls every row for that top-level value; `.loc[("Seoul", "Elec")]` drills into a specific combination. In practice, `.reset_index()` is almost always the next line, flattening the MultiIndex back into normal columns.  
**한글:** 둘 이상의 기준으로 그룹화하면 **MultiIndex**가 생깁니다 — 두 개(또는 그 이상)의 레벨이 겹겹이 쌓인 인덱스입니다. `.loc["Seoul"]`은 그 최상위 값에 해당하는 모든 행을 꺼내고, `.loc[("Seoul", "Elec")]`은 특정 조합으로 파고듭니다. 실무에서는 거의 항상 `.reset_index()`가 다음 줄에 와서 MultiIndex를 다시 일반 열로 평탄화합니다.

In [10]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["Seoul", "Busan", "Seoul", "Incheon", "Seoul", "Busan"],
    "category": ["Elec", "Cloth", "Elec", "Food", "Cloth", "Elec"],
    "sales": [320000, 85000, 150000, 43000, 210000, 95000],
})

multi = orders.groupby(["region", "category"])["sales"].agg(["sum", "count"])
print("MultiIndex result:")
print(multi)
print()

print("loc['Seoul'] -- every category within Seoul:")
print(multi.loc["Seoul"])
print()

print("loc[('Seoul', 'Elec')] -- one specific combination:")
print(multi.loc[("Seoul", "Elec")])
print()

print("reset_index() -- back to normal flat columns:")
print(multi.reset_index())

MultiIndex result:
                     sum  count
region  category               
Busan   Cloth      85000      1
        Elec       95000      1
Incheon Food       43000      1
Seoul   Cloth     210000      1
        Elec      470000      2

loc['Seoul'] -- every category within Seoul:
             sum  count
category               
Cloth     210000      1
Elec      470000      2

loc[('Seoul', 'Elec')] -- one specific combination:
sum      470000
count         2
Name: (Seoul, Elec), dtype: int64

reset_index() -- back to normal flat columns:
    region category     sum  count
0    Busan    Cloth   85000      1
1    Busan     Elec   95000      1
2  Incheon     Food   43000      1
3    Seoul    Cloth  210000      1
4    Seoul     Elec  470000      2


## Example 9 — Common Combo Patterns
*(Covers source section 7-11)*

**English:** Pattern A chains `agg()` + `assign()` (for a share-of-total %) + `sort_values()` into a one-page region report. Pattern B combines `transform()` with Boolean indexing to pull out only the rows performing *above* their own group's average — a filter that couldn't be written without first computing the group value via `transform()`.  
**한글:** 패턴 A는 `agg()` + `assign()`(전체 대비 비중 %) + `sort_values()`를 체이닝해서 한 페이지짜리 지역 리포트를 만듭니다. 패턴 B는 `transform()`과 Boolean indexing을 결합해서 자기 그룹 평균*보다* 높은 행만 뽑아냅니다 — 먼저 `transform()`으로 그룹 값을 계산하지 않고서는 쓸 수 없는 필터입니다.

In [11]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": [1,2,3,4,5,6,7,8,9,10],
    "region": ["Seoul","Busan","Seoul","Incheon","Seoul","Busan","Incheon","Seoul","Busan","Seoul"],
    "sales": [320000, 85000, 150000, 43000, 210000, 95000, 38000, 67000, None, 190000],
})

# Pattern A: agg + share% + sort -- a one-page region report
# 패턴 A: agg + 비중% + 정렬 -- 한 페이지짜리 지역 리포트
report = (
    orders.groupby("region").agg(
        total_sales = ("sales", "sum"),
        order_count = ("order_id", "count"),
    )
    .reset_index()
    .assign(share_pct=lambda x: (x["total_sales"] / x["total_sales"].sum() * 100).round(1))
    .sort_values("total_sales", ascending=False)
)
print("Pattern A -- region report:")
print(report)
print()

# Pattern B: transform + Boolean indexing -- rows above their OWN group's average
# 패턴 B: transform + Boolean indexing -- 자기 그룹 평균보다 높은 행
orders["region_avg"] = orders.groupby("region")["sales"].transform("mean")
above_avg = orders[orders["sales"] > orders["region_avg"]]
print("Pattern B -- orders beating their own region's average:")
print(above_avg[["order_id", "region", "sales", "region_avg"]])

Pattern A -- region report:
    region  total_sales  order_count  share_pct
2    Seoul     937000.0            5       78.2
0    Busan     180000.0            3       15.0
1  Incheon      81000.0            2        6.8

Pattern B -- orders beating their own region's average:
   order_id   region     sales  region_avg
0         1    Seoul  320000.0    187400.0
3         4  Incheon   43000.0     40500.0
4         5    Seoul  210000.0    187400.0
5         6    Busan   95000.0     90000.0
9        10    Seoul  190000.0    187400.0


## Example 10 (Practice) — Fill in the Blanks
*(Based on the practice exercise in source section 7-12)*

**English:** Fill in each `________` blank below. The code is syntactically valid Python, so it won't raise a `SyntaxError` — but it also won't print any result until every blank is correct (it will raise a runtime error instead, which is expected).
**한글:** 아래 `________` 빈칸을 채워보세요. 코드는 문법적으로 올바른 파이썬이라 `SyntaxError`는 나지 않지만, 모든 빈칸이 정확해지기 전까지는 결과가 출력되지 않습니다(대신 런타임 오류가 나는데, 이는 의도된 동작입니다).

In [13]:
import pandas as pd

emp = pd.DataFrame({
    "name": ["Minsu","Younghee","Junho","Seoyeon","Daehyun","Soyoung","Jihoon","Yerin","Jiwon","Areum"],
    "dept": ["Sales","Sales","Marketing","Engineering","Marketing","Engineering","Sales","Engineering","Marketing","Sales"],
    "salary": [4200000,3800000,3500000,5100000,4500000,5800000,4000000,4900000,4100000,3600000],
    "years": [3,5,2,7,6,9,4,8,3,2],
})

# 1. Department average salary + headcount, both in one agg() call
#    부서별 평균 연봉 + 인원수를 agg() 한 번으로
result1 = emp.groupby("dept").agg(
    avg_salary = ("salary", "mean"),
    headcount  = ("name", "count"),
).reset_index()

# 2. Each employee's salary compared to their OWN department's average
#    각 직원의 연봉을 자기 부서 평균과 비교
emp["dept_avg"] = emp.groupby("dept")["salary"].transform("mean")
emp["vs_avg"] = emp["salary"] - emp["dept_avg"]

# 3. The full row of the top earner in EACH department
#    각 부서에서 최고 연봉자의 전체 행
idx = emp.groupby("dept")["salary"].idxmax()
top_earners = emp.loc[idx][["dept", "name", "salary"]]

print(result1)
print(emp[["dept", "name", "salary", "dept_avg", "vs_avg"]])
print(top_earners)

          dept    avg_salary  headcount
0  Engineering  5.266667e+06          3
1    Marketing  4.033333e+06          3
2        Sales  3.900000e+06          4
          dept      name   salary      dept_avg         vs_avg
0        Sales     Minsu  4200000  3.900000e+06  300000.000000
1        Sales  Younghee  3800000  3.900000e+06 -100000.000000
2    Marketing     Junho  3500000  4.033333e+06 -533333.333333
3  Engineering   Seoyeon  5100000  5.266667e+06 -166666.666667
4    Marketing   Daehyun  4500000  4.033333e+06  466666.666667
5  Engineering   Soyoung  5800000  5.266667e+06  533333.333333
6        Sales    Jihoon  4000000  3.900000e+06  100000.000000
7  Engineering     Yerin  4900000  5.266667e+06 -366666.666667
8    Marketing     Jiwon  4100000  4.033333e+06   66666.666667
9        Sales     Areum  3600000  3.900000e+06 -300000.000000
          dept     name   salary
5  Engineering  Soyoung  5800000
4    Marketing  Daehyun  4500000
0        Sales    Minsu  4200000


### 💡 Hint / 힌트
`mean` · `count` · `transform` · `idxmax`

### ✅ Solution / 정답
*(Try solving it yourself first! / 먼저 스스로 풀어본 뒤에 확인하세요!)*

In [ ]:
import pandas as pd

emp = pd.DataFrame({
    "name": ["Minsu","Younghee","Junho","Seoyeon","Daehyun","Soyoung","Jihoon","Yerin","Jiwon","Areum"],
    "dept": ["Sales","Sales","Marketing","Engineering","Marketing","Engineering","Sales","Engineering","Marketing","Sales"],
    "salary": [4200000,3800000,3500000,5100000,4500000,5800000,4000000,4900000,4100000,3600000],
    "years": [3,5,2,7,6,9,4,8,3,2],
})

result1 = emp.groupby("dept").agg(
    avg_salary = ("salary", "mean"),
    headcount  = ("name", "count"),
).reset_index()

emp["dept_avg"] = emp.groupby("dept")["salary"].transform("mean")
emp["vs_avg"] = emp["salary"] - emp["dept_avg"]

idx = emp.groupby("dept")["salary"].idxmax()
top_earners = emp.loc[idx][["dept", "name", "salary"]]

print(result1)
print()
print(emp[["dept", "name", "salary", "dept_avg", "vs_avg"]])
print()
print(top_earners)

# Engineering has the highest average (5,266,667) despite having only 3 people --
# a small group can still out-earn a bigger one on average. That's exactly the kind
# of insight a bare row-count or a bare sum would have hidden.
# Engineering은 인원이 3명뿐인데도 평균이 가장 높음(5,266,667) -- 작은 그룹도
# 평균으로는 더 큰 그룹을 앞설 수 있음. 이는 단순 인원수나 단순 합계로는
# 보이지 않았을 인사이트입니다.

---
# ⚠️ Common Mistakes

### Mistake 1 — Forgetting `.reset_index()` after a GroupBy
**English:** `orders.groupby("region")["sales"].sum()` returns a Series with `region` as the **index**, not a column. Code written right after that assumes `result["region"]` exists will raise a `KeyError`, because `region` isn't a column yet.  
**한글:** `orders.groupby("region")["sales"].sum()`은 `region`이 열이 아니라 **인덱스**인 Series를 반환합니다. 바로 뒤에 `result["region"]`이 존재한다고 가정하는 코드를 쓰면 `KeyError`가 발생하는데, `region`이 아직 열이 아니기 때문입니다.

**✅ Fix / 해결법:**  
Make `.reset_index()` a reflex right after almost any GroupBy — it turns the group key(s) back into normal columns.  
거의 모든 GroupBy 뒤에는 반사적으로 `.reset_index()`를 붙이세요 — 그룹 키를 다시 일반 열로 되돌립니다.

### Mistake 2 — Reaching for `agg()` when the goal actually needs `transform()`
**English:** `groupby("region")["sales"].agg("mean")` produces **one row per region** — you can't directly attach that group average back onto each of the original 10,000 rows to compute "this order vs. its region's average" without first merging it back manually.  
**한글:** `groupby("region")["sales"].agg("mean")`은 **지역당 한 행**을 만듭니다 — "이 주문 vs 지역 평균"을 계산하려고 원본 만 개 행에 그 그룹 평균을 직접 다시 붙일 수 없습니다, 수동으로 다시 합치지 않는 한.

**✅ Fix / 해결법:**  
Whenever the group's aggregate needs to sit *next to* every original row (not replace them), use `.transform()` instead of `.agg()` — same syntax, but the row count stays the same.  
그룹의 집계값이 원본 행을 대체하는 게 아니라 모든 원본 행 *옆에* 있어야 한다면, `.agg()` 대신 `.transform()`을 사용하세요 — 문법은 같지만 행 개수가 그대로 유지됩니다.

### Mistake 3 — Treating `.count()` and `.size` (or `len()`) as interchangeable
**English:** `.count()` on a column **excludes `NaN` values**, so different columns in the same `.agg()` call can report different counts. `.size` (a property, no parentheses) or `len(group)` counts every row in the group regardless of missing data. Mixing them up produces a subtly wrong headcount.  
**한글:** 열에 대한 `.count()`는 **`NaN` 값을 제외**하므로, 같은 `.agg()` 호출 안에서도 열마다 다른 개수를 보고할 수 있습니다. `.size`(속성, 괄호 없음)나 `len(group)`은 결측 여부와 상관없이 그룹의 모든 행을 셉니다. 이 둘을 헷갈리면 미묘하게 틀린 인원수가 나옵니다.

**✅ Fix / 해결법:**  
Use `.size` (or a column you know has no missing values) when you want a true row count; use `.count()` on a specific column when you specifically want "how many non-missing values."  
진짜 행 개수를 원한다면 `.size`(또는 결측치가 없다고 확실한 열)를 사용하고, 특정 열에서 "결측이 아닌 값이 몇 개인지"를 원한다면 그 열에 `.count()`를 사용하세요.

---
# 💡 Tips
Useful tips or shortcuts / 유용한 팁과 단축법

- Chain `.reset_index()` onto a `groupby().agg()` result before doing anything else with it — it's the difference between a usable DataFrame and a Series with an unusual index.  
`groupby().agg()` 결과에 다른 작업을 하기 전에 `.reset_index()`를 체이닝하세요 — 사용 가능한 DataFrame과 특이한 인덱스를 가진 Series의 차이를 만듭니다.
- Reach for named aggregation (`.agg(x=("col","func"))`) the moment you need more than one summary statistic, or different statistics for different columns — it's clearer than chaining separate `.groupby()` calls.  
요약 통계가 두 개 이상 필요하거나 열마다 다른 통계가 필요한 순간 named aggregation(`.agg(x=("col","func"))`)을 사용하세요 — 별도의 `.groupby()` 호출을 체이닝하는 것보다 명확합니다.
- `transform()` is the tool whenever a calculation needs "this row's value compared to *its group's* aggregate" (% of total, rank within group, vs-average) — `agg()` alone can't do this because it drops the original rows.  
계산이 "이 행의 값 vs *자기 그룹*의 집계값"을 필요로 할 때는(전체 대비 %, 그룹 내 순위, 평균 대비) `transform()`을 사용하세요 — `agg()`만으로는 원본 행을 없애버리므로 불가능합니다.
- `pd.Grouper(freq=...)` needs a `DatetimeIndex` first (`set_index()` on the date column) — it's the way to combine time-based grouping with an ordinary column like `region` in the same `groupby()` call.  
`pd.Grouper(freq=...)`는 먼저 `DatetimeIndex`가 필요합니다(날짜 열에 `set_index()`) — 이는 시간 기준 그룹화를 `region` 같은 일반 열과 같은 `groupby()` 호출에서 결합하는 방법입니다.

---
# 🔗 Related Concepts

```
Column Creation & Transformation    (Section 6 -- the columns you built there become today's groupby() keys)
    ↓
Aggregation & GroupBy                 ← you are here / 지금 여기 (Section 7)
    ↓
Merge & Concatenation                (Section 8 -- combining a groupby summary back with the original table)
    ↓
Pivot Table & Reshape                (Section 9 -- unstack() turns a MultiIndex groupby result into a cross-tab instantly)
    ↓
Time Series -> BA Techniques
```

*How is today's topic connected to other concepts?*

**English:** A `spend_tier` column built with `pd.qcut()` in Section 6 is a perfectly ordinary column to `groupby()` today — nearly every real GroupBy groups by something *derived*, not something that arrived in the raw file. Looking ahead, Section 8's `merge()` is often how a shrunk `agg()` summary gets reattached to the original, full-size table, and Section 9's `.unstack()` turns exactly the kind of MultiIndex result from Example 8 into a wide cross-tab in one call.

**한글:** 6번 섹션에서 `pd.qcut()`으로 만든 `spend_tier` 열은 오늘 `groupby()`하기에 완벽하게 평범한 열입니다 — 실제 GroupBy는 거의 항상 원본 파일에 있던 것이 아니라 *파생된* 것을 기준으로 그룹화합니다. 앞을 내다보면, 8번 섹션의 `merge()`는 종종 축소된 `agg()` 요약을 원본의 전체 크기 테이블에 다시 붙이는 방법이고, 9번 섹션의 `.unstack()`은 Example 8에서 본 것과 정확히 같은 종류의 MultiIndex 결과를 한 번의 호출로 넓은 크로스탭으로 바꿉니다.

---
# 💼 Business Example
*How would a Business Analyst use this?*

**Scenario / 시나리오**

**English:** A regional sales director wants a one-page snapshot: total revenue and order count by region with each region's contribution %, plus the single biggest order in each region.

**한글:** 지역 영업 이사가 한 페이지짜리 스냅샷을 원합니다: 지역별 총매출과 주문 건수, 각 지역의 기여도 %, 그리고 지역별로 가장 큰 단일 주문.

**To Do / 할 일**
- [x] Build the region summary with named aggregation  
named aggregation으로 지역 요약 만들기
- [x] Add each region's share of the company-wide total  
각 지역의 전사 대비 비중 추가하기
- [x] Find the single biggest order per region with `idxmax()`  
`idxmax()`로 지역별 가장 큰 단일 주문 찾기
- [x] Sort the summary from largest region to smallest  
요약을 가장 큰 지역부터 작은 지역 순으로 정렬하기

In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": [1,2,3,4,5,6,7,8,9,10],
    "region": ["Seoul","Busan","Seoul","Incheon","Seoul","Busan","Incheon","Seoul","Busan","Seoul"],
    "sales": [320000, 85000, 150000, 43000, 210000, 95000, 38000, 67000, 120000, 190000],
})

# Region summary with contribution % / 기여도 %가 포함된 지역 요약
summary = (
    orders.groupby("region").agg(
        total_sales = ("sales", "sum"),
        order_count = ("order_id", "count"),
    )
    .reset_index()
    .assign(share_pct=lambda x: (x["total_sales"] / x["total_sales"].sum() * 100).round(1))
    .sort_values("total_sales", ascending=False)
)
print("Region summary:")
print(summary)
print()

# Biggest single order per region / 지역별 가장 큰 단일 주문
top_idx = orders.groupby("region")["sales"].idxmax()
top_orders = orders.loc[top_idx][["region", "order_id", "sales"]].sort_values("sales", ascending=False)
print("Biggest order per region:")
print(top_orders)

---
# 📝 Summary
*Write today's concept in 3~5 sentences.*

**English**
GroupBy follows split → apply → combine: split rows into groups by one or more keys, apply an aggregate to each group, and combine the results into a new table — almost always followed by `.reset_index()`. Named aggregation (`.agg(name=("col","func"))`) applies a different summary statistic per column in a single call — the single most-used real-world pattern. `transform()` broadcasts a group's aggregate back onto every original row (for a share %, a rank, a vs-average comparison), while `filter()` keeps or drops entire groups based on a group-level condition — neither shrinks the row count the way `agg()` does. `shift()` compares a value to the previous row within its own group (month-over-month), and `expanding()` accumulates from the very first row instead of a fixed window. `idxmax()` / `idxmin()` return the row **index** of a group's extreme value, and `pd.Grouper` groups by a time frequency alongside an ordinary column. Grouping by more than one key produces a MultiIndex, which `.loc[]` can navigate directly or `reset_index()` can flatten back to normal columns.

**한글**
GroupBy는 분할 → 적용 → 결합을 따릅니다: 하나 이상의 기준으로 행을 그룹으로 나누고, 각 그룹에 집계를 적용한 뒤, 결과를 새 테이블로 결합합니다 — 거의 항상 `.reset_index()`가 뒤따릅니다. Named aggregation(`.agg(name=("col","func"))`)은 한 번의 호출로 열마다 다른 요약 통계를 적용합니다 — 실무에서 가장 많이 쓰이는 패턴입니다. `transform()`은 그룹의 집계값을 모든 원본 행에 다시 broadcast하고(비중 %, 그룹 내 순위, 평균 대비 비교), `filter()`는 그룹 수준 조건에 따라 그룹 전체를 유지하거나 제거하는데 — 둘 다 `agg()`처럼 행 개수를 줄이지 않습니다. `shift()`는 값을 자기 그룹 안에서 이전 행과 비교하고(전월 대비), `expanding()`은 고정된 윈도우 대신 첫 행부터 누적합니다. `idxmax()` / `idxmin()`은 그룹 극값의 행 **인덱스**를 반환하고, `pd.Grouper`는 일반 열과 함께 시간 빈도로 그룹화합니다. 둘 이상의 기준으로 그룹화하면 MultiIndex가 생기는데, `.loc[]`으로 직접 탐색하거나 `reset_index()`로 다시 일반 열로 평탄화할 수 있습니다.

---
# 📌 One Sentence Summary
Today's topic in ONE sentence.

> GroupBy is split → apply → combine — break rows into groups, summarize (or transform, or filter) each one, and get a result shaped for exactly the question being asked, whether that's "one row per group" or "the group's context attached to every original row."

> GroupBy는 분할 → 적용 → 결합입니다 — 행을 그룹으로 나누고, 각각을 요약(또는 변환, 또는 필터링)한 뒤, 질문이 요구하는 정확한 형태의 결과를 얻습니다 — "그룹당 한 행"이든 "모든 원본 행에 붙은 그룹의 맥락"이든.

---
# ❓ Review Questions

**Q1.** Why does `orders.groupby("region")["sales"].sum()` usually need a `.reset_index()` right after it?
**Q1.** `orders.groupby("region")["sales"].sum()` 뒤에는 왜 보통 바로 `.reset_index()`가 필요한가요?

**Q2.** What does `.agg(total=("sales","sum"), count=("sales","count"))` do that a plain `.groupby("region")["sales"].sum()` can't?
**Q2.** `.agg(total=("sales","sum"), count=("sales","count"))`은 일반 `.groupby("region")["sales"].sum()`이 할 수 없는 무엇을 하나요?

**Q3.** What's the key difference in row count between the result of `.agg()` and the result of `.transform()`?
**Q3.** `.agg()`의 결과와 `.transform()`의 결과는 행 개수 면에서 핵심적으로 어떻게 다른가요?

**Q4.** What does `orders.groupby("region")["sales"].idxmax()` return — the biggest sales value, or something else?
**Q4.** `orders.groupby("region")["sales"].idxmax()`는 무엇을 반환하나요 — 가장 큰 매출 값인가요, 아니면 다른 무언가인가요?

**Q5.** What structure does `orders.groupby(["region","category"])["sales"].sum()` produce, and what's the fastest way to turn it back into a normal flat DataFrame?
**Q5.** `orders.groupby(["region","category"])["sales"].sum()`은 어떤 구조를 만들며, 이를 다시 평범한 flat DataFrame으로 되돌리는 가장 빠른 방법은 무엇인가요?

---
*📅 Try answering these again in a few days. / 며칠 뒤에 다시 답해보세요.*